# Harmonia — LoRA Fine-tuning

Fine-tunes LLaMA 3.2 3B on the Harmonia MIDI dataset using LoRA.

**Runtime:** A100 GPU required (Colab Pro). Training takes ~2–4 hours.

Prerequisites:
- Run `01_preprocess.ipynb` first and confirm the dataset is on HuggingFace Hub
- Accept the LLaMA 3.2 license on HuggingFace: https://huggingface.co/meta-llama/Llama-3.2-3B
- Have a HuggingFace token with read access to gated models

## Setup

In [ ]:
# Confirm we have a GPU
import torch

assert torch.cuda.is_available(), "No GPU found — switch to A100 runtime"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR      = "/content/drive/MyDrive/harmonia"
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints"

!mkdir -p {CHECKPOINT_DIR}

In [ ]:
!pip install transformers peft datasets accelerate bitsandbytes trl -q

In [ ]:
# ---- FILL THESE IN BEFORE RUNNING ----
HF_TOKEN        = "hf_..."                                # HuggingFace token (read + write)
HF_DATASET_REPO = "your-username/harmonia-midi"           # Dataset pushed in 01_preprocess
HF_MODEL_REPO   = "your-username/harmonia-llama-3.2-3b"  # Where to push the fine-tuned model
BASE_MODEL      = "meta-llama/Llama-3.2-3B"
# ---------------------------------------

## Step 1 — Load base model with 4-bit quantization

4-bit quantization (QLoRA) keeps the model footprint small enough to fit on a single A100
while still producing a high-quality fine-tune.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
model.config.use_cache = False

print("Model loaded.")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 2 — Load dataset

Streams the dataset from HuggingFace Hub. Each example is a text string with a
conditioning prompt followed by REMI tokens.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(HF_DATASET_REPO, token=HF_TOKEN, split="train")
print(f"Dataset size: {len(dataset):,} examples")

# Preview a sample
print("\nSample (first 400 chars):")
print(dataset[0]["text"][:400])

## Step 3 — Train

`DataCollatorForCompletionOnlyLM` masks the conditioning prompt from the loss —
the model only trains to predict tokens after `<MIDI_START>`, not to reproduce the prompt.

Checkpoints save to Google Drive every 1000 steps so a dropped session doesn't lose progress.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# Only train on tokens after <MIDI_START>
collator = DataCollatorForCompletionOnlyLM(
    response_template="<MIDI_START>",
    tokenizer=tokenizer,
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      # effective batch = 32
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=3,                 # keep only the 3 latest checkpoints
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    optim="paged_adamw_8bit",           # memory-efficient optimiser for QLoRA
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=1024,
)

trainer.train()

In [ ]:
# Manual checkpoint save — run this any time you want to snapshot mid-training
trainer.save_model(f"{CHECKPOINT_DIR}/manual-snapshot")
print("Snapshot saved to Drive.")

## Step 4 — Merge LoRA weights and push to Hub

Merges the LoRA adapters back into the base model weights and pushes the full
model to HuggingFace Hub. This is what the inference service will load.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Reload base model in full precision for merging
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,
)

merged = PeftModel.from_pretrained(base, f"{CHECKPOINT_DIR}/manual-snapshot")
merged = merged.merge_and_unload()

print("Merged. Pushing to Hub...")
merged.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN, private=True)
print(f"Done — model at https://huggingface.co/{HF_MODEL_REPO}")

In [ ]:
# Quick generation test — confirm the model produces MIDI tokens
prompt = "Compose a melancholic slow piece in A minor at 65 BPM featuring Piano, Bass.\n<MIDI_START>\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = merged.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(out[0], skip_special_tokens=True)
print(generated)